# Занятие 1. Введение в CNN. Процесс обучения в PyTorch

## Цели занятия:
- Освоить базовый пайплайн работы с данными в PyTorch
- Реализовать свёрточную нейронную сеть с нуля
- Написать полный цикл обучения модели
- Получить первую обученную модель с accuracy > 85%

---

## Шаг 1. Настройка окружения

**Задание:** Проверьте доступность GPU и зафиксируйте random seed для воспроизводимости.

In [ ]:
# Проверка доступности GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Device: CPU")

In [ ]:
# Фиксация random seed для воспроизводимости
import random
import numpy as np

def set_seed(seed=42):
    """Fix random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("Random seed fixed: 42")

---
## Шаг 2. Загрузка данных

**Задание:** Загрузите датасет FashionMNIST и создайте DataLoader'ы.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# TODO: Определите трансформации для тренировочных и тестовых данных
# Подсказка: используйте transforms.Compose с ToTensor() и Normalize()
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# TODO: Загрузите FashionMNIST
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

# TODO: Создайте DataLoader'ы
# ВАЖНО: num_workers=2 может вызвать ошибки в Windows или некоторых версиях Colab
# Если возникнут ошибки, установите num_workers=0
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0  # Changed from 2 for compatibility
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
# Визуализация нескольких примеров
import matplotlib.pyplot as plt

# Получаем один батч
images, labels = next(iter(train_loader))

# Денормализация для визуализации
def denormalize(tensor, mean=0.5, std=0.5):
    """Denormalize tensor for visualization."""
    return tensor * std + mean

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i].squeeze())
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {train_dataset.classes[labels[i]]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Шаг 3. Определение модели CNN

**Задание:** Реализуйте архитектуру свёрточной нейронной сети.

### Архитектура:
- 3 свёрточных блока (Conv2d + ReLU + MaxPool2d)
- 2 полносвязных слоя
- Dropout для регуляризации

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    """Simple CNN for FashionMNIST classification."""
    
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layers
        # Input: 1x28x28
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # -> 32x28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # -> 64x14x14
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # -> 128x7x7
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        # After 3 pooling operations: 28 -> 14 -> 7 -> 3
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Conv block 1: 28x28 -> 14x14
        x = self.pool(F.relu(self.conv1(x)))
        
        # Conv block 2: 14x14 -> 7x7
        x = self.pool(F.relu(self.conv2(x)))
        
        # Conv block 3: 7x7 -> 3x3
        x = self.pool(F.relu(self.conv3(x)))
        
        # Flatten
        x = x.view(-1, 128 * 3 * 3)
        
        # FC layers
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        
        return x

# Создание модели
model = SimpleCNN(num_classes=10)
print(model)

# Подсчёт количества параметров
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
## Шаг 4. Настройка обучения

**Задание:** Настройте функцию потерь, оптимизатор и scheduler.

In [ ]:
# Выбор устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Перенос модели на устройство
model = model.to(device)

# Функция потерь
criterion = nn.CrossEntropyLoss()

# Оптимизатор
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Scheduler для изменения learning rate
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

print("Training setup completed!")
print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Scheduler: StepLR (step_size=5, gamma=0.5)")

---
## Шаг 5. Функции обучения и валидации

**Задание:** Реализуйте функции для одной эпохи обучения и валидации.

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader, criterion, optimizer, device):
    """Train model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    """Validate model for one epoch."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating', leave=False):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

print("Training functions defined!")

---
## Шаг 6. Цикл обучения

**Задание:** Запустите полный цикл обучения модели.

In [ ]:
import matplotlib.pyplot as plt

# Хранение истории обучения
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# Параметры обучения
num_epochs = 15
best_acc = 0.0

print(f"Starting training on {device}...")
print("="*50)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Обучение
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Валидация
    val_loss, val_acc = validate_epoch(
        model, test_loader, criterion, device
    )
    
    # Scheduler step
    scheduler.step()
    
    # Сохранение истории
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Сохранение лучшей модели
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model_cnn.pth')
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

print("\n" + "="*50)
print(f"Training completed!")
print(f"Best Validation Accuracy: {best_acc:.2f}%")

---
## Шаг 7. Визуализация результатов обучения

**Задание:** Постройте графики loss и accuracy.

In [ ]:
# Графики обучения
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss Curves', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy Curves', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Results:")
print(f"Best Train Accuracy: {max(history['train_acc']):.2f}%")
print(f"Best Val Accuracy: {max(history['val_acc']):.2f}%")

---
## Шаг 8. Визуализация предсказаний

**Задание:** Загрузите лучшую модель и визуализируйте предсказания.

In [ ]:
# Загрузка лучшей модели
model.load_state_dict(torch.load('best_model_cnn.pth', map_location=device))
model.eval()

# Классы FashionMNIST
classes = train_dataset.classes

# Получение батча тестовых данных
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# Предсказания
with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# Визуализация 10 изображений
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i].cpu().squeeze())
    ax.imshow(img, cmap='gray')
    
    true_label = classes[labels[i].item()]
    pred_label = classes[predicted[i].item()]
    
    color = 'green' if labels[i] == predicted[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 9. Визуализация карт признаков

**Задание:** Визуализируйте активации первого свёрточного слоя.

In [ ]:
def visualize_feature_maps(model, image, layer_name='conv1'):
    """Visualize feature maps from a specific layer."""
    model.eval()
    
    # Hook для получения активаций
    activations = {}
    
    def get_activation(name):
        def hook(module, input, output):
            activations[name] = output.detach()
        return hook
    
    # Регистрация hook
    getattr(model, layer_name).register_forward_hook(get_activation(layer_name))
    
    # Forward pass
    with torch.no_grad():
        model(image.unsqueeze(0).to(device))
    
    # Получение активаций
    feature_maps = activations[layer_name].cpu().squeeze()
    
    # Визуализация
    num_maps = min(feature_maps.shape[0], 32)  # Show max 32 maps
    n_cols = 8
    n_rows = (num_maps + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 1.5))
    
    for i, ax in enumerate(axes.flat):
        if i < num_maps:
            ax.imshow(feature_maps[i], cmap='viridis')
        ax.axis('off')
    
    plt.suptitle(f'Feature Maps - {layer_name} (showing {num_maps} of {feature_maps.shape[0]})', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'feature_maps_{layer_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Визуализация карт первого свёрточного слоя
sample_image, _ = test_dataset[0]
print(f"Input image shape: {sample_image.shape}")
visualize_feature_maps(model, sample_image, 'conv1')

---
## Домашнее задание

1. **Дообучить модель до accuracy > 88% на валидации**
   - Попробуйте увеличить количество эпох
   - Измените архитектуру (добавьте больше слоёв)

2. **Поэкспериментировать с гиперпараметрами**
   - Learning rate: попробуйте 0.0001, 0.001, 0.01
   - Batch size: попробуйте 32, 64, 128
   - Оптимизатор: сравните Adam и SGD

3. **Добавить сохранение истории обучения в CSV-файл**

4. **Подготовить краткий отчёт**
   - Описание изменений в архитектуре
   - Таблица с метриками
   - Выводы о влиянии параметров на качество

---
## Контрольные вопросы

1. Объясните разницу между Conv2d и Linear слоями.
2. Зачем нужна нормализация входных данных?
3. Как работает механизм backpropagation?
4. Почему используем Dropout в полносвязных слоях?
5. Как интерпретировать графики train/val loss?

In [ ]:
# Место для экспериментов с домашним заданием
# TODO: Добавьте ваши эксперименты здесь